# Campagne CPU TSA LUT

Notebook autonome de campagne CPU, même style que `Campagne_CPU_TSA.ipynb`,
mais avec l'algorithme TSA LUT intégré directement (sans dépendance externe).

In [3]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
    HAS_DISPLAY = True
except Exception:
    HAS_DISPLAY = False

try:
    from ipywidgets import Dropdown, IntText, SelectMultiple, VBox, Label, Output
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False

# ============================================================
# META / CAMPAGNE
# ============================================================
EPS = 1e-12
HEADER_RE = re.compile(
    r"PRN=(?P<prn>-?\d+)"
    r".*SNR=(?P<snr>-?\d+(?:\.\d+)?)dB"
    r".*Doppler=(?P<doppler>-?\d+(?:\.\d+)?)Hz"
    r".*CA_SHIFT=(?P<ca>-?\d+)chips",
    re.IGNORECASE,
)

# ============================================================
# TSA LUT constants (from provided algorithm)
# ============================================================
FS = 11999000.0
N = 11999
NB_PHASES = 1023
FC = 3563000.0

NB_DOPPLER_COARSE = 41
DOPPLER_BIN_COARSE = 500
PRECISION = 8
COARSE_TOPK = 16
SEUIL_VARIANCE_COARSE = 0.039

NB_DOPPLER_FINE = 21
DOPPLER_BIN_FINE = 250
MAX_NUM_PEAKS_FS = 16
SEUIL_VARIANCE_FINE = 0.04

DDS_LUT_BITS = 10
DDS_LUT_SIZE = 1 << DDS_LUT_BITS
DDS_PHASE_BITS = 32

PRN_VARIANTS = 3
CS_REF_RANK = 5
FS_REF_RANK = 4
MIN_PEAK_SEP = 16
MIN_DOPPLER_SEP_COARSE = DOPPLER_BIN_COARSE
MIN_DOPPLER_SEP_FINE = DOPPLER_BIN_FINE

FS_THRESHOLD_LOW_SCALE = 0.85
FS_GLOBAL_PEAK_GUARD_RATIO = 1.20
FS_LOCAL_DOMINANCE_MIN = 1.00
FS_LOCAL_PHASE_WIN = 1
FS_LOCAL_DOPPLER_WIN = DOPPLER_BIN_FINE
FINE_REFINE_DOPPLER_MAX = 2 * DOPPLER_BIN_FINE
FINE_REFINE_PHASE_MAX = 64

FINE_TAU_TILE = 16
ENABLE_FINE = True

# ============================================================
# DDS LUT loading (strict)
# ============================================================
def _parse_lut_block(header_text, lut_name):
    pattern = rf"static\s+const\s+osc_t\s+{lut_name}\s*\[\s*DDS_LUT_SIZE\s*\]\s*=\s*\{{(.*?)\}};"
    m = re.search(pattern, header_text, flags=re.S)
    if not m:
        raise ValueError(f"Block {lut_name} not found in dds_lut_rom.h")

    block = m.group(1)
    values = []
    for line in block.splitlines():
        mm = re.search(r"\(osc_t\)\s*([+-]?[0-9]*\.?[0-9]+(?:[eE][+-]?[0-9]+)?)", line)
        if mm:
            values.append(float(mm.group(1)))
    if len(values) != DDS_LUT_SIZE:
        raise ValueError(f"{lut_name}: expected {DDS_LUT_SIZE} values, got {len(values)}")
    return np.array(values, dtype=np.float64)


def load_dds_luts_from_header(header_path="dds_lut_rom.h"):
    p = Path(header_path)
    if not p.is_file():
        raise FileNotFoundError(f"{header_path} not found")
    txt = p.read_text(encoding="utf-8", errors="ignore")
    sin_lut = _parse_lut_block(txt, "DDS_SIN_LUT")
    cos_lut = _parse_lut_block(txt, "DDS_COS_LUT")
    return sin_lut, cos_lut


DDS_SIN_LUT, DDS_COS_LUT = load_dds_luts_from_header("dds_lut_rom.h")
print("[INFO] DDS LUT loaded from dds_lut_rom.h (strict mode, HLS-consistent).")

# ============================================================
# TSA LUT helpers
# ============================================================
def hz_to_phase_inc(fd):
    freq_total = -(int(FC) + int(fd))
    num = freq_total * (1 << DDS_PHASE_BITS)
    if num >= 0:
        return num // int(FS)
    return -((-num) // int(FS))


def dds_mix_subsample(signal, indices, fd, pass_idx):
    M = len(indices)
    phase_inc = hz_to_phase_inc(fd)
    phase_acc = (phase_inc * pass_idx) % (1 << 32)
    phase_step = (phase_inc * PRECISION) % (1 << 32)

    phases = (phase_acc + np.arange(M, dtype=np.int64) * phase_step) % (1 << 32)
    lut_idx = ((phases >> (DDS_PHASE_BITS - DDS_LUT_BITS)) & (DDS_LUT_SIZE - 1)).astype(int)

    cos_v = DDS_COS_LUT[lut_idx]
    sin_v = DDS_SIN_LUT[lut_idx]
    x = signal[indices]
    return x * cos_v, -(x * sin_v)


def dds_mix_full(signal, fd):
    phase_inc = hz_to_phase_inc(fd)
    phases = (np.arange(N, dtype=np.int64) * phase_inc) % (1 << 32)
    lut_idx = ((phases >> (DDS_PHASE_BITS - DDS_LUT_BITS)) & (DDS_LUT_SIZE - 1)).astype(int)

    cos_v = DDS_COS_LUT[lut_idx]
    sin_v = DDS_SIN_LUT[lut_idx]
    x = signal[:N]
    return x * cos_v, -(x * sin_v)


def build_prn_variants(prn):
    base = prn.copy()
    half_plus = 0.5 * (prn + np.roll(prn, -1))
    half_minus = 0.5 * (prn + np.roll(prn, +1))

    signs = np.zeros((PRN_VARIANTS, 2 * N), dtype=np.float64)
    for v, arr in enumerate([base, half_plus, half_minus]):
        s = np.where(arr >= 0, 1.0, -1.0)
        signs[v, :N] = s
        signs[v, N:] = s
    return signs


def build_tau_start_table():
    taus = np.arange(NB_PHASES)
    return (taus.astype(np.int64) * N // NB_PHASES).astype(int)


def build_stsa_indices():
    indices = []
    counts = []
    for p in range(PRECISION):
        idx = np.arange(p, N, PRECISION)
        indices.append(idx)
        counts.append(len(idx))
    return indices, counts


def phase_circular_dist(a, b):
    d = abs(int(a) - int(b))
    wrap = NB_PHASES - d
    return min(d, wrap)


def doppler_abs_dist(a, b):
    return abs(int(a) - int(b))


def insert_peak(peak_list, doppler, phase, power, max_size):
    entry = {"doppler": int(doppler), "phase": int(phase), "power": float(power)}
    if len(peak_list) < max_size:
        peak_list.append(entry)
    else:
        min_idx = min(range(len(peak_list)), key=lambda i: peak_list[i]["power"])
        if power > peak_list[min_idx]["power"]:
            peak_list[min_idx] = entry


def select_spaced_topk(peaks, k_target, phase_sep, doppler_sep):
    sorted_peaks = sorted(peaks, key=lambda p: p["power"], reverse=True)
    selected = []
    for pk in sorted_peaks:
        if len(selected) >= k_target:
            break
        far = True
        for sel in selected:
            if (
                phase_circular_dist(pk["phase"], sel["phase"]) < phase_sep
                and doppler_abs_dist(pk["doppler"], sel["doppler"]) < doppler_sep
            ):
                far = False
                break
        if far:
            selected.append(pk)
    return selected


def nvar_from_p1_pk(p1, pk):
    if p1 <= 0:
        return 0.0
    r = pk / p1
    m = (1.0 + r) * 0.5
    d1 = 1.0 - m
    d2 = r - m
    var = 0.5 * (d1 * d1 + d2 * d2)
    return max(var, 0.0)


def coarse_search(signal, prn_sign_double, tau_start_tbl, threshold_coarse=SEUIL_VARIANCE_COARSE, compare_mode=False):
    stsa_indices, stsa_counts = build_stsa_indices()

    doppler_offsets = np.array([
        DOPPLER_BIN_COARSE * (d - (NB_DOPPLER_COARSE - 1) // 2)
        for d in range(NB_DOPPLER_COARSE)
    ])

    pass_best_power = [0.0] * PRECISION
    pass_best_doppler = [0] * PRECISION
    pass_best_phase = [0] * PRECISION
    top_cs_pass = [[] for _ in range(PRECISION)]

    higherLowerPeaks = [0.0] * (PRECISION + 1)
    higherLowerPeaks[0] = 1.0
    ratios_head = 1
    ratios_valid = 1

    best_power = 0.0
    best_doppler = 0
    best_phase = 0
    coarse_detected = False
    coarse_var = 0.0

    pass_sign_factors = []
    for pass_idx in range(PRECISION):
        indices = stsa_indices[pass_idx]
        idx_matrix = tau_start_tbl[:, None] + indices[None, :]
        pass_sign_factors.append(prn_sign_double[idx_matrix])

    for d in range(NB_DOPPLER_COARSE):
        fd = int(doppler_offsets[d])
        accI = np.zeros(NB_PHASES)
        accQ = np.zeros(NB_PHASES)

        for pass_idx in range(PRECISION):
            indices = stsa_indices[pass_idx]

            mix_i, mix_q = dds_mix_subsample(signal, indices, fd, pass_idx)
            sign_factor = pass_sign_factors[pass_idx]

            accI += sign_factor @ mix_i
            accQ += sign_factor @ mix_q

            power_grid = accI ** 2 + accQ ** 2

            for tau in range(NB_PHASES):
                pwr = power_grid[tau]
                insert_peak(top_cs_pass[pass_idx], fd, tau, pwr, COARSE_TOPK)
                if pwr > pass_best_power[pass_idx]:
                    pass_best_power[pass_idx] = pwr
                    pass_best_doppler[pass_idx] = fd
                    pass_best_phase[pass_idx] = tau

    for pass_idx in range(PRECISION):
        pbp = pass_best_power[pass_idx]
        pbd = pass_best_doppler[pass_idx]
        pbph = pass_best_phase[pass_idx]

        if pbp > best_power:
            best_power = pbp
            best_doppler = pbd
            best_phase = pbph

        n_top = len(top_cs_pass[pass_idx])
        if n_top >= CS_REF_RANK:
            spaced = select_spaced_topk(top_cs_pass[pass_idx], CS_REF_RANK, MIN_PEAK_SEP, MIN_DOPPLER_SEP_COARSE)
            if len(spaced) >= CS_REF_RANK:
                p_highest = spaced[0]["power"]
                p_lowest = spaced[CS_REF_RANK - 1]["power"]
                ratio = (p_lowest / p_highest) if p_highest > 0 else 1.0

                higherLowerPeaks[ratios_head] = ratio
                ratios_head = (ratios_head + 1) % (PRECISION + 1)
                if ratios_valid < PRECISION + 1:
                    ratios_valid += 1

                vals = higherLowerPeaks[:ratios_valid]
                mean = sum(vals) / ratios_valid
                var = sum((v - mean) ** 2 for v in vals) / ratios_valid
                coarse_var = max(var, 0.0)

        if coarse_var > threshold_coarse:
            coarse_detected = True
            if not compare_mode:
                break

    return best_doppler, best_phase, best_power, coarse_detected, coarse_var


def fine_search(signal, prn_signs, tau_start_tbl,
                doppler_coarse, coarse_doppler_out, coarse_phase_out,
                coarse_peak_out, coarse_var_in,
                threshold_fine=SEUIL_VARIANCE_FINE,
                threshold_coarse_in=SEUIL_VARIANCE_COARSE):

    fine_doppler_count = NB_DOPPLER_FINE
    fine_tau_count = 192

    if threshold_coarse_in > 0:
        ultra_high = threshold_coarse_in * 3
        high_conf = threshold_coarse_in * 2
        mid_conf = threshold_coarse_in + threshold_coarse_in / 2
        if coarse_var_in >= ultra_high:
            fine_doppler_count, fine_tau_count = 7, 64
        elif coarse_var_in >= high_conf:
            fine_doppler_count, fine_tau_count = 9, 96
        elif coarse_var_in >= mid_conf:
            fine_doppler_count, fine_tau_count = 11, 128

    fine_tau_count = max(1, min(fine_tau_count, NB_PHASES))
    aligned = (fine_tau_count // FINE_TAU_TILE) * FINE_TAU_TILE
    if aligned < FINE_TAU_TILE:
        aligned = FINE_TAU_TILE
    fine_tau_count = aligned

    phase_center = coarse_phase_out
    fine_tau_begin = phase_center - fine_tau_count // 2
    fine_tau_begin = max(0, min(fine_tau_begin, NB_PHASES - fine_tau_count))

    center = NB_DOPPLER_FINE // 2
    active_half = fine_doppler_count // 2
    first_idx = center - active_half
    doppler_offsets_abs = [
        int(doppler_coarse) + (first_idx + d - center) * DOPPLER_BIN_FINE
        for d in range(fine_doppler_count)
    ]

    ListOfPeaksFS = [[] for _ in range(PRN_VARIANTS)]
    max_power_val = 0.0
    sum_power_val = 0.0

    fine_taus = np.arange(fine_tau_begin, fine_tau_begin + fine_tau_count)
    shifts = tau_start_tbl[fine_taus]
    sample_idx = np.arange(N, dtype=np.int64)
    idx_mat = shifts[:, None] + sample_idx[None, :]

    for d_idx in range(fine_doppler_count):
        fd = doppler_offsets_abs[d_idx]
        mix_i, mix_q = dds_mix_full(signal, fd)

        for v in range(PRN_VARIANTS):
            prn_v = prn_signs[v]
            sign_factor = prn_v[idx_mat]

            I_all = sign_factor @ mix_i
            Q_all = sign_factor @ mix_q

            power_all = I_all ** 2 + Q_all ** 2

            sum_power_val += np.sum(power_all)
            local_max = np.max(power_all)
            if local_max > max_power_val:
                max_power_val = local_max

            best_local_idx = np.argmax(power_all)
            best_tau = int(fine_taus[best_local_idx])
            insert_peak(ListOfPeaksFS[v], fd, best_tau, power_all[best_local_idx], MAX_NUM_PEAKS_FS)

    highest_peak = [None] * PRN_VARIANTS
    lowest_peak = [None] * PRN_VARIANTS
    has_peak = [False] * PRN_VARIANTS

    for v in range(PRN_VARIANTS):
        spaced = select_spaced_topk(ListOfPeaksFS[v], FS_REF_RANK, MIN_PEAK_SEP, MIN_DOPPLER_SEP_FINE)
        if len(spaced) >= FS_REF_RANK:
            highest_peak[v] = spaced[0]
            lowest_peak[v] = spaced[FS_REF_RANK - 1]
            has_peak[v] = True

    higher_var = 0.0
    best_var_variant = -1
    global_peak = None
    global_peak_variant = -1

    for v in range(PRN_VARIANTS):
        if has_peak[v]:
            var_p = nvar_from_p1_pk(highest_peak[v]["power"], lowest_peak[v]["power"])
            if best_var_variant < 0 or var_p > higher_var:
                higher_var = var_p
                best_var_variant = v
            if global_peak_variant < 0 or highest_peak[v]["power"] > (global_peak["power"] if global_peak else 0):
                global_peak = highest_peak[v]
                global_peak_variant = v

    best_peak = highest_peak[best_var_variant] if best_var_variant >= 0 else None

    def compute_local_comp(ref_pk):
        if ref_pk is None:
            return 0.0
        comp = 0.0
        for v in range(PRN_VARIANTS):
            for pk in ListOfPeaksFS[v]:
                same = (
                    pk["phase"] == ref_pk["phase"]
                    and pk["doppler"] == ref_pk["doppler"]
                    and pk["power"] == ref_pk["power"]
                )
                if not same:
                    d_ph = phase_circular_dist(ref_pk["phase"], pk["phase"])
                    d_dp = doppler_abs_dist(ref_pk["doppler"], pk["doppler"])
                    if d_ph <= FS_LOCAL_PHASE_WIN and d_dp <= FS_LOCAL_DOPPLER_WIN:
                        if pk["power"] > comp:
                            comp = pk["power"]
        return comp

    local_comp_best = compute_local_comp(best_peak)
    local_comp_global = compute_local_comp(global_peak)

    def dominance(pk, lc):
        if pk is None or pk["power"] <= 0:
            return 0.0
        return pk["power"] / (lc if lc > 0 else 1.0)

    dom_best = dominance(best_peak, local_comp_best)
    dom_global = dominance(global_peak, local_comp_global)

    global_guard = (
        best_var_variant >= 0 and global_peak_variant >= 0 and global_peak_variant != best_var_variant
        and global_peak is not None and best_peak is not None
        and global_peak["power"] > best_peak["power"] * FS_GLOBAL_PEAK_GUARD_RATIO
    )

    chosen_peak = best_peak
    if global_peak_variant >= 0 and global_peak is not None:
        if global_guard and dom_best >= FS_LOCAL_DOMINANCE_MIN and dom_global < FS_LOCAL_DOMINANCE_MIN:
            chosen_peak = best_peak
        else:
            chosen_peak = global_peak

    if chosen_peak is not None:
        fine_dopp_dist = doppler_abs_dist(chosen_peak["doppler"], coarse_doppler_out)
        fine_phase_dist = phase_circular_dist(chosen_peak["phase"], coarse_phase_out)
        coherent = (fine_dopp_dist <= FINE_REFINE_DOPPLER_MAX and fine_phase_dist <= FINE_REFINE_PHASE_MAX)
    else:
        coherent = False

    dom_chosen = dom_global if (chosen_peak is not None and global_peak is not None and chosen_peak["power"] == global_peak["power"]) else dom_best
    dom_ok = dom_chosen >= FS_LOCAL_DOMINANCE_MIN
    threshold_high = threshold_fine
    var_pass_high = higher_var > threshold_high
    fine_refine_ok = (chosen_peak is not None) and var_pass_high

    if fine_refine_ok and chosen_peak is not None:
        doppler_out = chosen_peak["doppler"]
        phase_out = chosen_peak["phase"]
        peak_out = chosen_peak["power"]
        peak_fine_out = chosen_peak["power"]
    else:
        doppler_out = coarse_doppler_out
        phase_out = coarse_phase_out
        peak_out = coarse_peak_out
        peak_fine_out = coarse_peak_out

    fine_var = higher_var

    return (
        doppler_out, phase_out, peak_out, fine_refine_ok,
        fine_var, peak_fine_out, fine_doppler_count, fine_tau_count,
        max_power_val, sum_power_val
    )


def acquisition_stsa(signal, prn, threshold_coarse=SEUIL_VARIANCE_COARSE, threshold_fine=SEUIL_VARIANCE_FINE, compare_mode=False):
    prn_signs = build_prn_variants(prn)
    tau_start_tbl = build_tau_start_table()

    prn_base_double = prn_signs[0]

    c_doppler, c_phase, c_peak, c_detected, c_var = coarse_search(
        signal, prn_base_double, tau_start_tbl, threshold_coarse, compare_mode
    )

    result = {
        "coarse_doppler": c_doppler,
        "coarse_phase": c_phase,
        "coarse_peak": c_peak,
        "coarse_detected": c_detected,
        "coarse_var": c_var,
    }

    # Fallback: si coarse_var est "proche" du seuil (>= 85%), lancer fine_search
    # pour ne pas rater une detection borderline (ex: Doppler hors grille a -20dB).
    # Si coarse_var est clairement faible, on abandonne sans lancer fine_search.
    # Aucun seuil these modifie. Decision finale = fine_refine_ok.
    COARSE_FALLBACK_RATIO = 0.85
    coarse_promising = (c_var >= threshold_coarse * COARSE_FALLBACK_RATIO)

    if not c_detected and not coarse_promising:
        result.update({
            "doppler": c_doppler,
            "phase": c_phase,
            "peak": c_peak,
            "detected": False,
            "fine_var": 0.0,
            "fine_refine_ok": False,
            "fine_doppler_count": 0,
            "fine_tau_count": 0,
            "max_power": 0,
            "mean_power": 0,
        })
        return result

    (
        f_doppler, f_phase, f_peak, f_refine_ok,
        f_var, _f_peak_fine, f_dop_count, f_tau_count,
        f_max_power, f_sum_power,
    ) = fine_search(
        signal, prn_signs, tau_start_tbl,
        c_doppler, c_doppler, c_phase, c_peak, c_var,
        threshold_fine, threshold_coarse,
    )

    total_corr = f_dop_count * f_tau_count * PRN_VARIANTS
    mean_power = f_sum_power / total_corr if total_corr > 0 else 0.0

    result.update({
        "doppler": f_doppler,
        "phase": f_phase,
        "peak": f_peak,
        "detected": bool(f_refine_ok),
        "fine_var": f_var,
        "fine_refine_ok": f_refine_ok,
        "fine_doppler_count": f_dop_count,
        "fine_tau_count": f_tau_count,
        "max_power": f_max_power,
        "mean_power": mean_power,
    })
    return result

# ============================================================
# Campaign helpers
# ============================================================
def parse_meta_from_header(csv_file):
    with Path(csv_file).open("r", encoding="utf-8", errors="ignore") as f:
        first = f.readline().strip()
    m = HEADER_RE.search(first)
    if not m:
        raise ValueError(f"Metadonnees introuvables: {csv_file}")
    return {
        "prn": int(m.group("prn")),
        "snr": float(m.group("snr")),
        "doppler": int(round(float(m.group("doppler")))),
        "ca_shift": int(m.group("ca")),
    }


def load_signal_pm1(csv_file):
    with Path(csv_file).open("r", encoding="utf-8", errors="ignore") as f:
        _ = f.readline()
        line = f.readline().strip()
        if not line:
            f.seek(0)
            line = f.readline().strip()
    data = np.fromstring(line, sep=",", dtype=np.float64)
    if data.size == 0:
        raise ValueError(f"Signal vide/format invalide: {csv_file}")
    data = np.where(data == 255, -1, data).astype(np.float64)
    return data


def load_prn_sequence(prn_dir, prn, n_samples):
    prn_dir = Path(prn_dir)
    candidates = [
        prn_dir / f"PRN-{prn}.csv", prn_dir / f"prn-{prn}.csv",
        prn_dir / f"PRN_{prn}.csv", prn_dir / f"prn_{prn}.csv",
    ]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        raise FileNotFoundError(f"PRN {prn} introuvable dans {prn_dir}")
    seq = np.genfromtxt(path, delimiter=",", dtype=np.float64)
    seq = np.atleast_1d(seq).reshape(-1)
    if seq.size < n_samples:
        raise ValueError(f"PRN trop court: {seq.size} < {n_samples}")
    return seq[:n_samples].astype(np.float64)


def get_available_prns(prn_dir):
    prn_dir = Path(prn_dir)
    prns = []
    for pat in ["PRN-*.csv", "prn-*.csv", "PRN_*.csv", "prn_*.csv"]:
        for f in prn_dir.glob(pat):
            m = re.search(r"(\d+)", f.stem)
            if m:
                prns.append(int(m.group(1)))
    return sorted(set(prns))


def collect_signal_files(signals_dir, signal_glob):
    files = sorted(Path(signals_dir).glob(signal_glob))
    valid = []
    for f in files:
        try:
            parse_meta_from_header(f)
            valid.append(f)
        except Exception:
            pass
    return valid


def resolve_export_dir(cfg):
    return Path(str(cfg["export_dir"]))


def cpu_tsa_lut_acquisition(signal, prn_seq, cfg):
    t0 = time.perf_counter()

    sig = np.asarray(signal[:N], dtype=np.float64)
    prn = np.asarray(prn_seq[:N], dtype=np.float64)

    result = acquisition_stsa(
        sig,
        prn,
        threshold_coarse=float(cfg.get("threshold_coarse", SEUIL_VARIANCE_COARSE)),
        threshold_fine=float(cfg.get("threshold_fine", SEUIL_VARIANCE_FINE)),
        compare_mode=False,
    )

    peak = float(result.get("peak", 0.0))
    mean_power = float(result.get("mean_power", 0.0))
    peak_ratio = peak / mean_power if mean_power > EPS else 0.0

    t1 = time.perf_counter()
    return {
        "method": "cpu_tsa_lut",
        "detected": int(bool(result.get("detected", False))),
        "doppler": int(result.get("doppler", 0)),
        "phase": int(result.get("phase", 0)),
        "peak": peak,
        "peak_ratio": peak_ratio,
        "time_s": float(t1 - t0),
        "time_kernel_s": float(t1 - t0),
        "time_prepare_s": 0.0,
        "time_dma_s": 0.0,
        "time_ip_wait_s": 0.0,
        "time_end_to_end_s": float(t1 - t0),
    }


def run_method(method_name, signal, prn_seq, cfg):
    if method_name == "cpu_tsa_lut":
        return cpu_tsa_lut_acquisition(signal, prn_seq, cfg)
    raise ValueError(f"Methode inconnue: {method_name}")


def build_winner_table(df_raw, cfg):
    winner_rows = []
    for (file_name, method_name), group in df_raw.groupby(["file", "method"], dropna=False):
        valid = group[group["status"] == "ok"].sort_values("peak_ratio", ascending=False).reset_index(drop=True)
        if valid.empty:
            e = group.iloc[0]
            winner_rows.append({"file": file_name, "method": method_name, "status": e["status"], "error_message": e.get("error_message", "")})
            continue

        top1 = valid.iloc[0]
        top2_ratio = float(valid.iloc[1]["peak_ratio"]) if len(valid) > 1 else np.nan
        winner_margin_db = float(10.0 * np.log10((float(top1["peak_ratio"]) + EPS) / (top2_ratio + EPS))) if not np.isnan(top2_ratio) else np.nan
        robust_detected = int((float(top1["peak_ratio"]) >= float(cfg["detection_ratio_min"])) and (np.isnan(winner_margin_db) or winner_margin_db >= float(cfg["winner_margin_db_min"])))

        doppler_err = int(abs(int(top1["doppler_meas_hz"]) - int(top1["doppler_true_hz"])))
        phase_err = int(phase_circular_dist(int(top1["phase_meas_chip"]), int(top1["phase_true_chip"])))
        prn_ok = int(int(top1["prn_tested"]) == int(top1["prn_injected"]))
        doppler_ok = int(doppler_err <= int(cfg["doppler_tol_hz"]))
        phase_ok = int(phase_err <= int(cfg["phase_tol_chip"]))
        strict_success = int((robust_detected == 1) and (prn_ok == 1) and (doppler_ok == 1) and (phase_ok == 1))
        false_alarm = int((robust_detected == 1) and (prn_ok == 0))
        miss = int(robust_detected == 0)

        winner_rows.append({
            "file": file_name,
            "method": method_name,
            "status": "ok",
            "error_message": "",
            "snr_db": float(top1["snr_db"]),
            "prn_injected": int(top1["prn_injected"]),
            "prn_winner": int(top1["prn_tested"]),
            "doppler_true_hz": int(top1["doppler_true_hz"]),
            "doppler_meas_hz": int(top1["doppler_meas_hz"]),
            "doppler_err_hz": doppler_err,
            "phase_true_chip": int(top1["phase_true_chip"]),
            "phase_meas_chip": int(top1["phase_meas_chip"]),
            "phase_err_chip": phase_err,
            "peak": float(top1["peak"]),
            "peak_ratio": float(top1["peak_ratio"]),
            "second_peak_ratio": top2_ratio,
            "winner_margin_db": winner_margin_db,
            "time_s": float(top1["time_s"]),
            "classic_detected": int(top1["detected"]),
            "robust_detected": robust_detected,
            "pd_flag": robust_detected,
            "pfa_flag": false_alarm,
            "pm_flag": miss,
            "prn_ok": prn_ok,
            "doppler_ok": doppler_ok,
            "phase_ok": phase_ok,
            "strict_success": strict_success,
        })
    return pd.DataFrame(winner_rows)

# ============================================================
# Interactive menu (same style)
# ============================================================
DEFAULT_CONFIG = {
    "signals_dir": "signals",
    "prn_dir": "PRN",
    "signal_glob": "*.csv",
    "run_methods": ["cpu_tsa_lut"],
    "limit_signals": None,
    "prn_list": None,
    "fs_hz": FS,
    "if_hz": FC,
    "n_samples": N,
    "nb_phases": NB_PHASES,
    "fd_start": -10000,
    "fd_end": 10000,
    "fd_step": 500,
    "detection_ratio_min": 2.5,
    "winner_margin_db_min": 3.0,
    "doppler_tol_hz": 500,
    "phase_tol_chip": 3,
    "threshold_coarse": SEUIL_VARIANCE_COARSE,
    "threshold_fine": SEUIL_VARIANCE_FINE,
    "export_prefix": "validation_cpu_tsa_lut",
    "export_dir": "Resultats validation TSA",
    "export_methods_subdir": False,
    "resume_from_partial": True,
    "autosave_every_acq": 10,
    "prn_search_mode": "injected_plus_decoys",
    "pfa_decoy_count": 8,
    "pfa_decoy_seed": 42,
}

WIDGET_STATE = {
    "enabled": False,
    "limit_mode": None,
    "limit_value": None,
    "fd_mode": None,
    "fd_start": None,
    "fd_end": None,
    "fd_step": None,
    "prn_mode": None,
    "prn_selector": None,
    "prn_search_mode": None,
}


def collect_signal_files_for_ui(base_dir):
    base_dir = Path(base_dir)
    if not base_dir.exists():
        return []
    files = sorted(base_dir.glob("*.csv"))
    valid_files = []
    for file_path in files:
        try:
            with file_path.open("r", encoding="utf-8", errors="ignore") as handle:
                first = handle.readline().strip()
            if HEADER_RE.search(first):
                valid_files.append(file_path.name)
        except Exception:
            continue
    return valid_files


def get_available_prns_for_ui(base_dir):
    base_dir = Path(base_dir)
    if not base_dir.exists():
        return []
    prns = []
    patterns = ["PRN-*.csv", "prn-*.csv", "PRN_*.csv", "prn_*.csv"]
    for pattern in patterns:
        for file_path in base_dir.glob(pattern):
            match = re.search(r"(\d+)", file_path.stem)
            if match:
                prns.append(int(match.group(1)))
    return sorted(set(prns))


def get_current_config():
    cfg = dict(DEFAULT_CONFIG)
    if not WIDGET_STATE["enabled"]:
        return cfg

    cfg["limit_signals"] = None if int(WIDGET_STATE["limit_mode"].value) == 0 else int(WIDGET_STATE["limit_value"].value)

    if int(WIDGET_STATE["fd_mode"].value) == 0:
        cfg["fd_start"] = -10000
        cfg["fd_end"] = 10000
        cfg["fd_step"] = 500
    else:
        cfg["fd_start"] = int(WIDGET_STATE["fd_start"].value)
        cfg["fd_end"] = int(WIDGET_STATE["fd_end"].value)
        cfg["fd_step"] = int(WIDGET_STATE["fd_step"].value)

    if int(WIDGET_STATE["prn_mode"].value) == 0:
        cfg["prn_list"] = None
    else:
        cfg["prn_list"] = list(WIDGET_STATE["prn_selector"].value)

    psm = int(WIDGET_STATE["prn_search_mode"].value)
    cfg["prn_search_mode"] = "injected_only" if psm == 0 else ("injected_plus_decoys" if psm == 1 else "full_grid")
    return cfg


_available_signals = collect_signal_files_for_ui(DEFAULT_CONFIG["signals_dir"])
_available_prns = get_available_prns_for_ui(DEFAULT_CONFIG["prn_dir"])

print("=== CONFIGURATION INTERACTIVE ===")
print(f"Signaux trouves: {len(_available_signals)}")
print(f"PRN disponibles: {_available_prns}")

if HAS_WIDGETS and HAS_DISPLAY and _available_prns:
    limit_mode = Dropdown(options=[("Tous", 0), ("Limiter", 1)], value=0, description="Signaux")
    limit_value = IntText(value=min(10, max(1, len(_available_signals))), description="Nb")
    fd_mode = Dropdown(options=[("Standard", 0), ("Personnalise", 1)], value=0, description="Doppler")
    fd_start = IntText(value=DEFAULT_CONFIG["fd_start"], description="Fd debut")
    fd_end = IntText(value=DEFAULT_CONFIG["fd_end"], description="Fd fin")
    fd_step = IntText(value=DEFAULT_CONFIG["fd_step"], description="Fd step")
    prn_mode = Dropdown(options=[("Tous PRN", 0), ("Selection", 1)], value=0, description="PRN")
    prn_selector = SelectMultiple(options=[(str(p), p) for p in _available_prns], value=tuple(_available_prns[:1]), rows=min(10, len(_available_prns)), description="Liste")
    prn_search_mode = Dropdown(
        options=[("Injecte seulement", 0), ("Injecte + leurres", 1), ("Grille complete", 2)],
        value=1,
        description="Recherche",
    )
    config_output = Output()

    def refresh_config(*_):
        with config_output:
            config_output.clear_output(wait=True)
            cfg = get_current_config()
            print("=== CONFIG EFFECTIVE (selon menu) ===")
            print({
                "run_methods": cfg["run_methods"],
                "limit_signals": cfg["limit_signals"],
                "fd_start": cfg["fd_start"],
                "fd_end": cfg["fd_end"],
                "fd_step": cfg["fd_step"],
                "prn_list": cfg["prn_list"],
                "prn_search_mode": cfg["prn_search_mode"],
                "export_prefix": cfg["export_prefix"],
                "export_dir": cfg["export_dir"],
                "autosave_every_acq": cfg["autosave_every_acq"],
            })

    for w in [limit_mode, limit_value, fd_mode, fd_start, fd_end, fd_step, prn_mode, prn_selector, prn_search_mode]:
        w.observe(refresh_config, names="value")

    WIDGET_STATE.update({
        "enabled": True,
        "limit_mode": limit_mode,
        "limit_value": limit_value,
        "fd_mode": fd_mode,
        "fd_start": fd_start,
        "fd_end": fd_end,
        "fd_step": fd_step,
        "prn_mode": prn_mode,
        "prn_selector": prn_selector,
        "prn_search_mode": prn_search_mode,
    })

    display(VBox([
        Label("Choix de campagne CPU TSA LUT"),
        limit_mode,
        limit_value,
        fd_mode,
        fd_start,
        fd_end,
        fd_step,
        prn_mode,
        prn_selector,
        prn_search_mode,
        config_output,
    ]))
    refresh_config()
else:
    print("[INFO] Widgets indisponibles: utilisation de DEFAULT_CONFIG.")

CONFIG = get_current_config()
print("[INFO] La configuration effective se met a jour dans le panneau interactif ci-dessus.")


[INFO] DDS LUT loaded from dds_lut_rom.h (strict mode, HLS-consistent).
=== CONFIGURATION INTERACTIVE ===
Signaux trouves: 86
PRN disponibles: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]


[INFO] La configuration effective se met a jour dans le panneau interactif ci-dessus.


In [4]:
# ============================================================
# Run campaign
# ============================================================
cfg = get_current_config()
print("=== CONFIG UTILISEE POUR CE RUN ===")
print({
    "run_methods": cfg["run_methods"],
    "limit_signals": cfg["limit_signals"],
    "fd_start": cfg["fd_start"],
    "fd_end": cfg["fd_end"],
    "fd_step": cfg["fd_step"],
    "prn_list": cfg["prn_list"],
    "prn_search_mode": cfg["prn_search_mode"],
    "export_prefix": cfg["export_prefix"],
    "export_dir": cfg["export_dir"],
    "autosave_every_acq": cfg["autosave_every_acq"],
    "resume_from_partial": cfg.get("resume_from_partial", False),
})

signals_dir = Path(cfg["signals_dir"])
prn_dir = Path(cfg["prn_dir"])
signal_files = collect_signal_files(signals_dir, cfg["signal_glob"])
if cfg.get("limit_signals"):
    signal_files = signal_files[: int(cfg["limit_signals"])]
if not signal_files:
    raise FileNotFoundError("Aucun signal valide trouve dans signals/")

available_prns = get_available_prns(prn_dir)
selected_prns = [int(p) for p in cfg["prn_list"] if int(p) in available_prns] if cfg.get("prn_list") else available_prns

_n_decoy = int(cfg.get("pfa_decoy_count", 8))
_n_prn_per_sig = 1 + min(_n_decoy, max(0, len(selected_prns) - 1))
_n_acq = len(signal_files) * _n_prn_per_sig
print(f"Acquisitions prevues: {_n_acq}")

export_dir = resolve_export_dir(cfg)
export_dir.mkdir(parents=True, exist_ok=True)
prefix = str(cfg["export_prefix"])
raw_partial = export_dir / f"{prefix}_raw_partial.csv"

# Resume logic: load partial CSV if it exists
print(f"[DEBUG RESUME] resume_from_partial={cfg.get('resume_from_partial', False)}")
print(f"[DEBUG RESUME] raw_partial={raw_partial}")
print(f"[DEBUG RESUME] raw_partial.exists()={raw_partial.exists()}")
# Fichiers a re-tester et remplacer dans les resultats (sans toucher les autres)
# Laisser vide [] pour un resume normal
RETRY_FILES = [
    "gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n20_doppler-n6250Hz_ca-804.csv",
    "gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n20_doppler-n8750Hz_ca-804.csv",
]

# Cherche le partial meme si renomme avec " (1)" par Windows/OneDrive
_partial_candidates = [
    raw_partial,
    raw_partial.parent / (raw_partial.stem + " (1)" + raw_partial.suffix),
]
_partial_to_load = next((p for p in _partial_candidates if p.exists()), None)

_done_keys = set()
new_rows = []
if cfg.get("resume_from_partial", False) and _partial_to_load:
    _prev = pd.read_csv(_partial_to_load)
    if RETRY_FILES:
        _prev_keep = _prev[~_prev["file"].isin(RETRY_FILES)]
        _n_retry = len(_prev) - len(_prev_keep)
        print(f"[RETRY] {_n_retry} entrees exclues pour re-run: {RETRY_FILES}")
    else:
        _prev_keep = _prev
    new_rows = _prev_keep.to_dict(orient="records")
    for _, _r in _prev_keep.iterrows():
        _done_keys.add((str(_r["file"]), int(_r["prn_tested"])))
    print(f"[RESUME] {len(new_rows)} lignes chargees depuis {_partial_to_load.name}")
    print(f"[RESUME] Acquisitions restantes: ~{_n_acq - len(_done_keys)} sur {_n_acq}")
    del _prev, _prev_keep
else:
    print("[FRESH] Starting new campaign from scratch")

prn_cache = {}
_counter = len(new_rows)
_autosave_every = max(1, int(cfg.get("autosave_every_acq", 10)))

def _flush_partial_local():
    if new_rows:
        pd.DataFrame(new_rows).to_csv(raw_partial, index=False)

for i_sig, sig_file in enumerate(signal_files, 1):
    meta = parse_meta_from_header(sig_file)
    prn_injected = int(meta["prn"])
    doppler_true_hz = int(meta["doppler"])
    phase_true_chip = int(meta["ca_shift"] % int(cfg["nb_phases"]))
    snr_db = float(meta["snr"])

    signal = load_signal_pm1(sig_file)
    signal = np.pad(signal, (0, max(0, int(cfg["n_samples"]) - signal.size)), mode="edge")[: int(cfg["n_samples"])].astype(np.float64)

    _others = [int(p) for p in selected_prns if int(p) != int(prn_injected)]
    if _others:
        rng = np.random.default_rng(int(cfg.get("pfa_decoy_seed", 42)) + i_sig)
        pick = rng.choice(np.array(_others, dtype=np.int64), size=min(_n_decoy, len(_others)), replace=False)
        prns_this_signal = [int(prn_injected)] + sorted(int(x) for x in pick.tolist())
    else:
        prns_this_signal = [int(prn_injected)]

    for prn_tested in prns_this_signal:
        if (sig_file.name, prn_tested) in _done_keys:
            continue

        _prn_t0 = time.perf_counter()
        _prn_acq_count = 0

        if prn_tested not in prn_cache:
            prn_cache[prn_tested] = load_prn_sequence(prn_dir, prn_tested, int(cfg["n_samples"]))
        prn_seq = prn_cache[prn_tested]

        try:
            t_e2e0 = time.perf_counter()
            result = run_method("cpu_tsa_lut", signal, prn_seq, cfg)
            t_e2e1 = time.perf_counter()
            _counter += 1
            _prn_acq_count += 1
            print(f"[{_counter}/{_n_acq}] cpu_tsa_lut PRN={prn_tested} SNR={snr_db:g} DONE global={float(t_e2e1 - t_e2e0):.3f}s")
            new_rows.append({
                "file": sig_file.name,
                "method": "cpu_tsa_lut",
                "status": "ok",
                "error_message": "",
                "snr_db": snr_db,
                "prn_injected": prn_injected,
                "prn_tested": int(prn_tested),
                "doppler_true_hz": doppler_true_hz,
                "phase_true_chip": phase_true_chip,
                "detected": int(result["detected"]),
                "doppler_meas_hz": int(result["doppler"]),
                "phase_meas_chip": int(result["phase"]),
                "peak": float(result["peak"]),
                "peak_ratio": float(result["peak_ratio"]),
                "time_s": float(result["time_s"]),
                "time_kernel_s": float(result.get("time_kernel_s", result["time_s"])),
                "time_prepare_s": float(result.get("time_prepare_s", np.nan)),
                "time_dma_s": 0.0,
                "time_ip_wait_s": 0.0,
                "time_end_to_end_s": float(t_e2e1 - t_e2e0),
            })
        except Exception as exc:
            _counter += 1
            _prn_acq_count += 1
            print(f"[{_counter}/{_n_acq}] cpu_tsa_lut PRN={prn_tested} SNR={snr_db:g} ERROR: {exc}")
            new_rows.append({
                "file": sig_file.name,
                "method": "cpu_tsa_lut",
                "status": "error",
                "error_message": str(exc),
                "snr_db": snr_db,
                "prn_injected": prn_injected,
                "prn_tested": int(prn_tested),
                "doppler_true_hz": doppler_true_hz,
                "phase_true_chip": phase_true_chip,
                "detected": 0,
                "doppler_meas_hz": np.nan,
                "phase_meas_chip": np.nan,
                "peak": np.nan,
                "peak_ratio": np.nan,
                "time_s": np.nan,
                "time_kernel_s": np.nan,
                "time_prepare_s": np.nan,
                "time_dma_s": np.nan,
                "time_ip_wait_s": np.nan,
                "time_end_to_end_s": np.nan,
            })

        if (_counter % _autosave_every) == 0:
            _flush_partial_local()

        _prn_elapsed = time.perf_counter() - _prn_t0
        print(
            f"[PRN-TOTAL] file={sig_file.name} PRN={prn_tested} acquisitions={_prn_acq_count} "
            f"global_elapsed={_prn_elapsed:.3f}s"
        )

_flush_partial_local()
raw_results = pd.DataFrame(new_rows)
winner_results = build_winner_table(raw_results, cfg)
raw_results.to_csv(export_dir / f"{prefix}_raw_results.csv", index=False)
winner_results.to_csv(export_dir / f"{prefix}_winner_results.csv", index=False)

print("Exports:")
print(export_dir / f"{prefix}_raw_results.csv")
print(export_dir / f"{prefix}_winner_results.csv")
display(raw_results.head())
display(winner_results.head())


=== CONFIG UTILISEE POUR CE RUN ===
{'run_methods': ['cpu_tsa_lut'], 'limit_signals': None, 'fd_start': -10000, 'fd_end': 10000, 'fd_step': 500, 'prn_list': None, 'prn_search_mode': 'injected_plus_decoys', 'export_prefix': 'validation_cpu_tsa_lut', 'export_dir': 'Resultats validation TSA', 'autosave_every_acq': 10, 'resume_from_partial': True}
Acquisitions prevues: 774
[DEBUG RESUME] resume_from_partial=True
[DEBUG RESUME] raw_partial=Resultats validation TSA/validation_cpu_tsa_lut_raw_partial.csv
[DEBUG RESUME] raw_partial.exists()=True
[RETRY] 18 entrees exclues pour re-run: ['gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n20_doppler-n6250Hz_ca-804.csv', 'gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n20_doppler-n8750Hz_ca-804.csv']
[RESUME] 756 lignes chargees depuis validation_cpu_tsa_lut_raw_partial.csv
[RESUME] Acquisitions restantes: ~18 sur 774
[757/774] cpu_tsa_lut PRN=25 SNR=-20 DONE global=57.737s
[PRN-TOTAL] file=gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n20_doppler-n6250Hz_ca-804.csv PRN=2

,file,method,status,error_message,snr_db,prn_injected,prn_tested,doppler_true_hz,phase_true_chip,detected,doppler_meas_hz,phase_meas_chip,peak,peak_ratio,time_s,time_kernel_s,time_prepare_s,time_dma_s,time_ip_wait_s,time_end_to_end_s
0,gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_dop...,cpu_tsa_lut,ok,NaN,-10.0,25,25,-10000,804,1,-10000,804,4.281982e+06,88.128146,38.879518,38.879518,0.0,0.0,0.0,38.879630
1,gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_dop...,cpu_tsa_lut,ok,NaN,-10.0,25,1,-10000,804,0,-8000,104,1.768821e+05,0.000000,36.677451,36.677451,0.0,0.0,0.0,36.677535
2,gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_dop...,cpu_tsa_lut,ok,NaN,-10.0,25,2,-10000,804,0,-3000,227,1.775363e+05,0.000000,36.664614,36.664614,0.0,0.0,0.0,36.664696
3,gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_dop...,cpu_tsa_lut,ok,NaN,-10.0,25,9,-10000,804,0,4500,171,1.748841e+05,0.000000,36.510993,36.510993,0.0,0.0,0.0,36.511074
4,gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_dop...,cpu_tsa_lut,ok,NaN,-10.0,25,11,-10000,804,0,3000,271,1.781287e+05,0.000000,36.459088,36.459088,0.0,0.0,0.0,36.459175


,file,method,status,error_message,snr_db,prn_injected,prn_winner,doppler_true_hz,doppler_meas_hz,doppler_err_hz,...,time_s,classic_detected,robust_detected,pd_flag,pfa_flag,pm_flag,prn_ok,doppler_ok,phase_ok,strict_success
0,gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_dop...,cpu_tsa_lut,ok,,-10.0,25,25,-10000,-10000,0,...,38.879518,1,1,1,0,0,1,1,1,1
1,gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_dop...,cpu_tsa_lut,ok,,-10.0,25,25,-1250,-1250,0,...,39.549671,1,1,1,0,0,1,1,1,1
2,gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_dop...,cpu_tsa_lut,ok,,-10.0,25,25,-2500,-2500,0,...,41.803452,1,1,1,0,0,1,1,1,1
3,gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_dop...,cpu_tsa_lut,ok,,-10.0,25,25,-3750,-3750,0,...,38.546496,1,1,1,0,0,1,1,1,1
4,gps-1ms_prn-25_fs-11MHz_if-3563kHz_snr-n10_dop...,cpu_tsa_lut,ok,,-10.0,25,25,-5000,-5000,0,...,38.993224,1,1,1,0,0,1,1,1,1
